# Day 6: 两块小数字理解 Online Softmax

2026-11-07用户确认开始Day 6，个人预测与执行结果记入 [今日学习记录](../docs/DAY06_STUDY_LOG.md)。
配套讲义：`docs/DAY06_ONLINE_SOFTMAX.md`。使用CPU，先推导数学，不测试GPU速度。
目标不是背公式，而是理解旧累积量为什么要换基准。
字母、配置映射和轴顺序统一见 [符号与张量形状规范](../docs/NOTATION.md)。
该规范第7节含特殊符号的中文读法、英文名称和公式朗读；配套讲义第6–7节解释FlashAttention收益与边界。

## 先认字母与张量轴

| 符号 | 含义 |
| --- | --- |
| Q / K / V | Query查询向量 / Key匹配向量 / Value被汇集的内容向量 |
| B / Hq / Hkv | 序列数量 / Query头数 / K与V各自的头数 |
| T_q / T_kv | 本轮Query位置数 / 当前可读K/V位置数 |
| D_k / D_v | 每头Q/K特征维度 / 每头V特征维度 |
| E | Embedding / Model Dimension：本课程的嵌入及残差主干宽度，常记d_model |
| F | Feed-Forward Intermediate Dimension：FFN中间宽度，常记d_ff |
| S / P / O | softmax前分数 / 归一化权重 / 每头加权输出 |

FFN = Feed-Forward Network，前馈神经网络。E/F是本课程维度符号，不是所有框架统一的缩写。
K/V头按Query头对齐后：`S=(Q @ K.transpose(-2,-1))/sqrt(D_k)+Mask`，
`P=softmax(S,dim=-1)`，`O=P @ V`。
S/P形状为`[B,Hq,T_q,T_kv]`，O为`[B,Hq,T_q,D_v]`。
完整Self-Attention Prefill才通常有`T_q=T_kv=T`，可简写`[B,H,T,T]`，其中H=Hq。
`T`不是Tensor，也不是某个token。它是sequence length，表示位置槽位数量。
可以这样记：token是内容单位，position是排队位置，T是位置数量，Tensor是装它们的数组。
Attention分数表是“提问者×资料卡片”：行数T_q是Query位置数，列数T_kv是可查询K/V位置数。
多模态时，文字位置可对应text token，图片位置可对应image patch，音频位置可对应time frame，视频位置可对应时空patch。
四维张量不要用`K.T`代替末两轴转置。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/online_attention.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from torch.nn import functional as F
from learning.online_attention import online_softmax_chunks, blocked_attention
scores = torch.tensor([1., 2., 3., 4.], dtype=torch.float64)
values = torch.tensor([10., 20., 30., 40.], dtype=torch.float64)

### 先观察一次形状：从方阵Prefill到单行Decode

只用随机CPU张量验证字母与轴，不加载模型、不测速；示例为MHA，Hq=Hkv=2，未加Mask。
注意D_k=4而D_v=5也合法。Query特征轴参与点积后消失，输出最后一轴来自V。
以下Decode只验证形状，不声称复现真实模型的缓存更新或位置编码。

In [ ]:
def show_attention_shapes(query_length, kv_length):
    batch, query_heads, key_dim, value_dim = 1, 2, 4, 5
    generator = torch.Generator().manual_seed(123)
    query = torch.randn(batch, query_heads, query_length, key_dim, generator=generator)
    key = torch.randn(batch, query_heads, kv_length, key_dim, generator=generator)
    value = torch.randn(batch, query_heads, kv_length, value_dim, generator=generator)
    transposed_key = key.transpose(-2, -1)
    score = (query @ transposed_key) / key_dim**0.5
    probability = score.softmax(dim=-1)
    output = probability @ value
    for name, tensor in (('Q', query), ('K', key), ('K_transposed', transposed_key),
                         ('V', value), ('S', score), ('P', probability), ('O', output)):
        print(name, tuple(tensor.shape))
    assert score.shape == (batch, query_heads, query_length, kv_length)
    assert output.shape == (batch, query_heads, query_length, value_dim)
    torch.testing.assert_close(probability.sum(-1), torch.ones_like(probability.sum(-1)))
    expected_output = F.scaled_dot_product_attention(query, key, value, dropout_p=0.0)
    torch.testing.assert_close(output, expected_output, atol=1e-6, rtol=1e-5)

print('完整Prefill形状：T_q=T_kv=3')
show_attention_shapes(3, 3)
print('已存8位置再输入1位置：T_q=1，T_kv=9')
show_attention_shapes(1, 9)

## 1. 正确答案与一种错误合并法

先计算完整softmax，再把两个块分别softmax、求输出、简单平均。
为什么简单平均错误？因为它强行让两块获得相同总权重。

In [ ]:
reference = scores.softmax(0) @ values
first = scores[:2].softmax(0) @ values[:2]
second = scores[2:].softmax(0) @ values[2:]
incorrect = (first + second) / 2
print('正确完整输出:', reference.item())
print('第1块自己的输出:', first.item())
print('第2块自己的输出:', second.item())
print('错误简单平均:', incorrect.item())
assert not torch.isclose(reference, incorrect)

## 2. 维护 m、l、o

这些是数学变量，不是有唯一展开的缩写。建议专业名称如下：

| 符号 | 英文名称 | 含义 |
| --- | --- | --- |
| m | Running Row Maximum | 当前行已见最大分数 |
| l / ℓ | Running Softmax Normalizer | 当前基准下的指数和，即分母 |
| o | Unnormalized Attention Output Accumulator | 指数加权Value和，即未归一化分子 |

`l/ℓ`读ell，不是数字1或log；`o`读oh，不是数字0；`α`读alpha，中文常说阿尔法。
每次最大值变大，旧l/o同时乘 `α=exp(m_old-m_new)`，再加入新块。
最终输出为o/l。可读成：“新ell等于阿尔法乘旧ell，加上当前块指数权重的和。”

In [ ]:
m = scores.new_tensor(-torch.inf)
denominator = scores.new_zeros(())
numerator = scores.new_zeros(())
for start in (0, 2):
    block_s, block_v = scores[start:start+2], values[start:start+2]
    new_m = torch.maximum(m, block_s.max())
    alpha = torch.exp(m - new_m)
    p = torch.exp(block_s - new_m)
    denominator = alpha * denominator + p.sum()
    numerator = alpha * numerator + p @ block_v
    m = new_m
    print(f'处理位置 {start}..{start+1}: m={m.item():.4f}, alpha={alpha.item():.4f}, l={denominator.item():.4f}, o={numerator.item():.4f}')
actual = numerator / denominator
print('Online结果:', actual.item())
torch.testing.assert_close(actual, reference)
print('与完整softmax对齐: passed')

### 2.1 o通常是向量：同时汇集两个内容通道

将Value改成`[[10,110],[20,120],[30,130],[40,140]]`，先预测最终输出。
同一组概率用于两个通道，第二个通道恒比第一个大100；因此输出也应相差100。
先用向量分子逐块汇集，再验证`m+ln(l)`等于自然对数logsumexp；l本身不是log。

In [ ]:
vector_values = torch.stack((values, values + 100), dim=-1)
vector_m = scores.new_tensor(-torch.inf)
vector_l = scores.new_zeros(())
vector_o = scores.new_zeros(vector_values.shape[-1])
for start in (0, 2):
    block_s = scores[start:start + 2]
    block_v = vector_values[start:start + 2]
    updated_m = torch.maximum(vector_m, block_s.max())
    rescale = torch.exp(vector_m - updated_m)
    block_weights = torch.exp(block_s - updated_m)
    vector_l = rescale * vector_l + block_weights.sum()
    vector_o = rescale * vector_o + block_weights @ block_v
    vector_m = updated_m
    print('m=', vector_m.item(), 'l=', vector_l.item(), 'o=', vector_o.tolist())
vector_output = vector_o / vector_l
vector_reference = scores.softmax(0) @ vector_values
torch.testing.assert_close(vector_output, vector_reference)
torch.testing.assert_close(vector_output, torch.stack((reference, reference + 100)))
torch.testing.assert_close(vector_m + vector_l.log(), torch.logsumexp(scores, dim=0))
assert vector_o.shape == (2,)
print('向量输出:', vector_output.tolist(), 'alignment: passed')
print('logsumexp = m + ln(l):', (vector_m + vector_l.log()).item())

## 3. 改块宽与大幅值输入

不要只测一个切分，也不要只测不会溢出的小分数。

In [ ]:
for chunk in (1, 2, 3, 8):
    probabilities, history = online_softmax_chunks(scores, chunk)
    torch.testing.assert_close(probabilities, scores.softmax(0))
    print('chunk:', chunk, 'output:', (probabilities @ values).item())
large = scores + 1000
stable, _ = online_softmax_chunks(large, 2)
torch.testing.assert_close(stable, scores.softmax(0))
assert torch.isfinite(stable).all()
print('整体增加1000后，稳定结果不变；直接exp可能溢出:', torch.isfinite(large.exp()).all().item())

## 4. 从一行分数回到Attention

真实Attention的分子o是向量；下面复用课程中的分块PyTorch参考，对照SDPA。
这不是生产GPU FlashAttention kernel，也不包含训练backward验收。

In [ ]:
torch.manual_seed(123)
q, k, v = [torch.randn(1, 2, 7, 8) for _ in range(3)]
expected = F.scaled_dot_product_attention(q, k, v, is_causal=True)
for q_block, k_block in ((1, 1), (3, 2), (4, 3)):
    result = blocked_attention(q, k, v, query_block=q_block, key_block=k_block)
    torch.testing.assert_close(result, expected, atol=1e-5, rtol=1e-5)
    print('Q/K block:', q_block, k_block, '最大误差:', (result - expected).abs().max().item())

## 5. FlashAttention为什么省中间存储与IO？

朴素分开执行：完整S写全局内存 → 读S做softmax → 完整P写回 → 读P/V算输出。
融合分块：加载Q/K/V块 → 片上计算局部分数并更新状态 → 继续下一块 → 写出输出与必要辅助信息。
m保证稳定，l保留全局归一化分母，o保留加权内容；无需为最终输出保留每个位置的概率。

下面只做理论数组算术，不分配大张量、不测显存峰值或速度。
单层完整S与单行m/l/o不是同一个统计范围，不能拿二者比值声称整模型节省倍数。

In [ ]:
batch, heads, value_width = 1, 16, 128
score_element_bytes, accumulator_element_bytes = 2, 4
row_state_bytes = (1 + 1 + value_width) * accumulator_element_bytes
score_sizes = []
for length in (4096, 8192, 16384):
    score_bytes = batch * heads * length * length * score_element_bytes
    score_sizes.append(score_bytes)
    print(f'T={length}: 单层单张FP16分数表={score_bytes / 2**20:.0f} MiB; '
          f'每行FP32累积状态={row_state_bytes} bytes（不含tile等资源）')
assert [size // 2**20 for size in score_sizes] == [512, 2048, 8192]
assert score_sizes[1] == 4 * score_sizes[0]
assert row_state_bytes == 520
query_tile, key_tile = 64, 128
tile_score_bytes = query_tile * key_tile * accumulator_element_bytes
tile_state_bytes = query_tile * row_state_bytes
assert tile_score_bytes == 32 * 2**10
assert tile_state_bytes == 33280
print('示意FP32分数tile:', tile_score_bytes / 2**10, 'KiB')
print('示意64行FP32累积状态:', tile_state_bytes / 2**10, 'KiB')
print('逻辑数组计数，不是实际SM资源占用，也不是完整工作内存。')

## 6. 正确性与性能要分开验收

- 数学：同一套稠密Attention，分块不等于稀疏近似，浮点顺序可造成误差。
- 计算：完整Prefill的主矩阵计算量仍是T平方阶，不因m/l/o变成线性。
- 状态：每个Query独立维护m/l/o，不能将其当作可跨Query复用的KV Cache。
- 内存：Q/K/V、输出、局部分数和必要辅助信息仍占空间；请求完整Attention map另计。
- 实现：本Notebook是CPU参考，PyTorch分块循环不是融合GPU FlashAttention kernel。
- 范围：online_softmax_chunks仍返回整条概率向量；blocked_attention只支持这里声明的简单形状与因果Mask。
- 训练：这里没有验证生产backward、dropout或通用全Mask行策略。
- 性能：SDPA可选不同后端；没有确认后端与单独Benchmark，就不能声称跑了FlashAttention或固定加速。

详细原理、数据流和面试追问见 [配套讲义](../docs/DAY06_ONLINE_SOFTMAX.md) 第6–9节。

## 自测与参考答案

**1. m/l/o叫什么？怎么读？** m是Running Row Maximum；l/ℓ读ell，是Running Softmax Normalizer；o读oh，是Unnormalized Attention Output Accumulator。它们是角色名称，不是缩写展开。

**2. 每块softmax后简单平均，丢了什么？** 各块对全局分母的相对贡献；它强行让两块权重一样，而正确概率质量不相同。

**3. 为什么旧分子和旧分母一起乘alpha？** 两者都按旧最大值计算指数，需要同步变换到新基准，才能维持同一比值。

**4. 为什么o通常是向量？** Value有D_v个内容通道，同一权重分别汇集各通道，最终o/l也有D_v个数。

**5. 好处到底是什么？** 不物化完整S/P，并通过GPU分块融合减少全局内存中间读写；不是删除Q/K匹配计算。

**6. 是否取消KV Cache？能复用上个Query的m/l/o吗？** 都不能这样理解。KV是历史表示，新Query的分数和累积状态一般不同。

**7. 为什么数学通过不代表生产GPU性能或训练backward？** 这里只验证forward与理论数组计数，没有生产融合kernel、反向实现或GPU计时。

**8. 单行520 bytes是否等于Attention只需520 bytes？** 不是，还要计其他Query行、Q/K/V、tile、输出和实现资源，且不能拿单行和整层作峰值比较。

**9. l是logsumexp吗？** 不是。在本教程自然指数定义下，logsumexp=m+ln(l)。

继续运行 `learning/day06_course.ipynb` 保存统一报告。个人验收后再进入Day 7。